# Building An agent that has memory and access

1. source: https://www.youtube.com/watch?v=6uG4_Ivv60E&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv, https://www.youtube.com/watch?v=CeEki_0mdGo&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=15
2. Note at the end we have function, loop_agent which is an agent without memory, I will also build an agant using langchain
2. Note: this is only for undersanding what the agent is:
3. framework of agent (steps that the agent does), note here we do not do the search by ourself like what did when building rag class:


    3.1. make a call to LLM (we pay here if we use openAI)
    
    3.2. LLM decides to invoke (perform) search('param') 
    
    3.3 We invoke the search, we have the results
    
    3.4. send the results back to the llm (another call), we pay again if we use open AI
    
    3.5. llm process the results
    
    3.6. llm decides to make another tool call
    
    3.7. send one more API request < --
    
    3.8. process, llm gives the answer (llm keep iterating)

4. The agent has 3 parts:
    
    4.1. Instruction, a developper prompt
    
    4.2. functions (tool) helper can call to carry out the task. For us that's only search.
    
    4.3. Memory, the message history. We append every prompt, every model output, and every tool 
    result. The agent reads this to know what it has already tried.

What is an agent, fundamentally?

An agent is just a loop:

User Question

      ↓

LLM thinks

      ↓

Need a tool?

      ↓

Use tool

      ↓

Observe result

      ↓

LLM thinks again

      ↓
      
Final answer

Plus:

Memory
Conversation History
State

## STEP 0, 1, 2, 3, & 4  for building an agent (has instructions, tools, and memory) using langchain

In [ ]:
# STEP 0, I need to check if the model behavious correctly (it has the ability to response correctly and uses tool, since not all the models use tool)
# from langchain_core.tools import tool
# from langchain_ollama import ChatOllama


# @tool
# def get_time() -> str:
#     """Get current time"""
#     return "10:00 AM"
# llm_to_test = ChatOllama(
#     model="qwen2.5:3b-instruct",#, # can not use since it does not have bind_tool "gemma3:4b" we can use llama3.1, .. gpt-5.4-mini
# )

# question = "what is the time"

# llm_to_test_with_tools = llm_to_test.bind_tools([get_time])

# response_from_llm_to_test = llm_to_test_with_tools.invoke(
#     question
# )

# print(response_from_llm_to_test.content) # the response is empty, because the model sees only the tool, I need to excute the tool and give the result back to llm
# print(response_from_llm_to_test.tool_calls)  # here it is not empty, so the model uses the tool
# print(response_from_llm_to_test.additional_kwargs)

In [ ]:
# NOTE the instruction is sooooooooooooooooo important
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from sqlitesearch import TextSearchIndex #sqliteindex 
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.messages import ToolMessage  # for tool call loop

###################################################################################
# Step 0, use sqlite elastic search to search a context from SQL DataBase
##################################################################################
index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    # keyword_fiels=['course'],
    db_path='feq.db' # set the db name
)

#############################
# Step 1, set LLM
############################
llm = ChatOllama(
    model="qwen2.5:3b-instruct",#, # can not use since it does not have bind_tool "gemma3:4b" we can use llama3.1, .. gpt-5.4-mini
)

############################################################
# Step 2, set TOOL (functions, acess to YT, ....)
#############################################################
# NOTE::::::: The description of the fuction, is so important
@tool
def search(query: str) -> dict[str, str]: # when I use agent_tools.add, then the type will be added automatically and I do not need to set it.
    """
    Search the LLM Zoomcamp FAQ database.

    Use this tool whenever the user asks questions about:
    - joining the course
    - deadlines
    - registration
    - lectures
    - homework
    - certificates
    - course content
    """  
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query=query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

#########################################################
# Step 3, LLM with TOOL
########################################################
llm_with_tools = llm.bind_tools([search])

####################################################################################
# STEP 4 — Messages (like OpenAI) to set the instruction and the question, ....
####################################################################################
instructions = """You MUST use the search tool before answering.""".strip()

question = "Search the FAQ and tell me if I can join the LLM Zoomcamp."

messages = [
    SystemMessage(content=instructions),
    HumanMessage(content=question)
]


# the following has to be in a loop
"""
    # here I tell the LLM to use a tool
    response = llm_with_tools.invoke(
    messages
    )

    print('Check if the LLM has access to the TOOL Search, it will not answer anything here')
    print(response.content) # the response is empty, because the model sees only the tool, I need to excute the tool and give the result back to llm
    print(response.tool_calls)  # here it is not empty, so the model uses the tool
    print(response.additional_kwargs)
    ####################################################################################
    # 🔁 STEP 5 — TOOL CALL LOOP (THIS is your OpenAI loop)
    ####################################################################################
    # everytime, I need to append the message to keep the converstion on, if I did not append then 
    messages.append(response)

"""

# # Note this has to be in a loop, till I finish all the tools
# tool_call = response.tool_calls[0]
# if tool_call['name'] == 'search':
#     # now I need to exectue the serach, and then give it to LLM
#     # search is not a function, it is a tool_kit now, with parameters, search_tool = {
#     # "type": "function",
#     # "name": "search",
#     # "description": "Search the FAQ database for entries matching the given query.",
#     # "parameters": {
#     #     "type": "object",
#     #     "properties": {
#     #         "query": {
#     #             "type": "string",
#     #             "description": "Search query text to look up in the course FAQ."
#     #         }
#     #     },
#     #     "required": ["query"],
#     #     "additionalProperties": False}
   
#     result = search.invoke(tool_call["args"])
#     # I need to give the result to LLM, and I do not forget to append messages since we need to keep the conversation on
#     # append to message
#     messages.append(
#                 ToolMessage(
#                     content=str(result),
#                     tool_call_id=tool_call["id"]
#                 )
#             )

llm_with_tools = llm.bind_tools([search])

instructions = """You MUST use the search tool before answering.""".strip()

question = "Search the FAQ and tell me if I can join the LLM Zoomcamp."

messages = [
    SystemMessage(content=instructions),
    HumanMessage(content=question)
]

it = 1
MAX_ITERATIONS = 10  # this is to prevent repeating, becaue maybe the LLM will stuck, repeat search

while True:

    # to exit from the loop, when I finish all the tool
    has_tool_call = False
    print(f"\niteration #{it}...")

    # tell the llm to use tools
    response = llm_with_tools.invoke(messages)
    
    # now, I need the conversation to be on, so I need to update the state of the message
    messages.append(response)

    # STOP CONDITION (same as OpenAI)
    if not has_tool_calls:
        break

    # LLM has tools only, now I need to get answer from the tool and update the staus of the message
    # note not all LLM have tool, small model they do not have tool like gemma3:4b
    if response.tool_calls:
        for tool_call in response.tool_calls:
            # get the result from the tool
            if tool_call['name'] == 'search': # if the tool_call is name 
                tool_result = search.invoke(tool_call['args'])
                # append the result to messages
                messages.append(
                    ToolMessage(
                    content=str(tool_result), # give the content
                    tool_call_id=tool_call["id"] # the tool id
                )
                )
        # we need to iterate again and feed the LLM with the result from the tool, feed the LLM with the result
        has_tool_calls = True
    else:
        # 5. FINAL ANSWER (no tool calls)
        print("ASSISTANT:")
        print(response.content)

    it += 1
    if it > MAX_ITERATIONS:
        print("Maximum iterations reached.")
        break

In [ ]:
print(search)

### Build An Agent Manulally

In [ ]:

MAX_ITERATIONS = 10 
INSTRUCTIONS = """You are a FAQ assistant for the LLM Zoomcamp.

Always use the search tool before answering.

Answer ONLY using information returned by the search tool.

Keep answers concise and directly address the user's question.

Do not add unrelated information.

If the answer is not found in the search results, say:
"I don't know.""".strip()


# INSTRUCTIONS = """
# You're a course teaching assistant.
# You're given a question from a course student and your task is to answer it.

# If you want to look up information, use the search function. 
# Use as many keywords from the user question as possible when making first requests.

# Make multiple searches. First perform search, analyze the results 
# and then perform more searches. 

# At the end, ask if there are other areas that the user wants to explore.
# """.strip()
question = "Search the FAQ and tell me if I can join the LLM Zoomcamp."


llm = ChatOllama(
    model="qwen2.5:3b-instruct",#, # can not use since it does not have bind_tool "gemma3:4b" we can use llama3.1, .. gpt-5.4-mini
)
llm_with_tools = llm.bind_tools([search])



def agent_loop(
    question,
    instruction=INSTRUCTIONS,
):
    messages = [
        SystemMessage(content=instruction),
        HumanMessage(content=question)
    ]
    it = 0

    while it < MAX_ITERATIONS:
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # No tool call -> final answer
        if not response.tool_calls:
            return response.content

        # Execute tools
        for tool_call in response.tool_calls:
            if tool_call["name"] == "search":
                tool_result = search.invoke(tool_call["args"]) # search(**tool_call["args"])
                messages.append(
                    ToolMessage(
                        content=str(tool_result),
                        tool_call_id=tool_call["id"]
                    )
                )
        it += 1

    print('iteration =', it)
    return response.content

agent_loop(
    question,
    instruction=INSTRUCTIONS,
)

In [ ]:
agent_loop(INSTRUCTIONS, "How do I run Olama locally?")

In [ ]:
agent_loop(instructions, "How do I run Olama locally?")

In [ ]:
agent_loop("what's queen gambit?")

### Build An Agent using langchain.agent
1. Tool-calling agent
2. RAG retrieval (RAG)
3. manual memory
4. langchain executer

In [ ]:
from sqlitesearch import TextSearchIndex
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain.tools import tool
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_ollama import ChatOllama
from langchain_core.messages import (
    HumanMessage,
    AIMessage,
)
index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    # keyword_fiels=['course'],
    db_path='feq.db' # set the db name
)

####  the search needs to return text
@tool
def search(query: str) -> dict[str, str]: # when I use agent_tools.add, then the type will be added automatically and I do not need to set it.
    """
    Search the LLM Zoomcamp FAQ database.

    Use this tool whenever the user asks questions about:
    - joining the course
    - deadlines
    - registration
    - lectures
    - homework
    - certificates
    - course content
    """  
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    search_results = index.search(
        query=query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")
    return "\n".join(lines).strip()




question= "Can I still join the LLM Zoomcamp?"

INSTRUCTIONS = """You are a FAQ assistant for the LLM Zoomcamp.

Always use the search tool before answering.

Answer ONLY using information returned by the search tool.

Keep answers concise and directly address the user's question.

Do not add unrelated information.

If the answer is not found in the search results, say:
"I don't know.""".strip()


chat_history = []
#  1. prompt setting
prompt= ChatPromptTemplate.from_messages([
    ("system", INSTRUCTIONS),
    MessagesPlaceholder("chat_history"), # memory, long-term conversation memory (YOU control this)
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"), # internal working memory, It is the agent’s thinking workspace include: tool inputs, tool outputs, tool calls it decided to make, interal resoning steps,  (LangChain controls this), internal agent reasoning ( It stores:previous tool calls, tool outputs,reasoning steps,intermediate actions. The agent writes into this automatically.)
])

# 2. create llm
llm = ChatOllama(
    model="qwen2.5:3b-instruct",
)

# 3. create an agent
tools = [search]

agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
)

response = agent_executor.invoke({
    "input": question,
    "chat_history": chat_history,
})

# save history
chat_history.append(HumanMessage(content=question))
chat_history.append(AIMessage(content=response["output"]))

response

In [ ]:
response['output']

# Wrap in a class

In [ ]:
class FAQAgent:

    def __init__(self, agent_executor):

        self.agent_executor = agent_executor
        self.chat_history = []

    def ask(self, question: str) -> str:

        # 1. call agent
        response = self.agent_executor.invoke({
            "input": question,
            "chat_history": self.chat_history,
        })

        answer = response["output"]

        # 2. update memory
        self.chat_history.append(HumanMessage(content=question))
        self.chat_history.append(AIMessage(content=answer))

        # 3. optional: limit memory size
        self.chat_history = self.chat_history[-10:]

        return answer
    
agent = create_tool_calling_agent(
    llm=llm,
    tools=[search],
    prompt=prompt,
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=[search],
    # verbose=True,
)

faq = FAQAgent(agent_executor)


print(faq.ask("Can I join the LLM Zoomcamp?"))

In [ ]:
print(faq.ask("Can I still  join the LLM Zoomcamp?"))

In [ ]:
print(faq.ask("Do I need confirmation email?"))

In [ ]:
import numpy as np
x = np.array([10,20,30,40,50,60])

np.argsort(x)[::-1]

In [ ]:
np.argsort(-x)

# HERE the Code is for testing the FAG agent ability to search

### check if the agent is able to use youTube channel

In [ ]:
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_ollama import ChatOllama
from langchain_community.tools import YouTubeSearchTool

# 1. Instantiate the official tool
youtube_tool = YouTubeSearchTool()

# 2. Simple, direct instructions to eliminate prompt layout edge cases
TEST_INSTRUCTIONS = """You are a video locator assistant. 
Always use the youtube_search tool to find relevant video links before answering the user's question."""


prompt = ChatPromptTemplate.from_messages([
    ("system", TEST_INSTRUCTIONS),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])


# 3. Setup your local LLM 
llm = ChatOllama(
    model="qwen2.5:3b-instruct",
    base_url="http://127.0.0.1:11434"   
)


# 4. Build the test agent
agent = create_tool_calling_agent(
    llm=llm,
    tools=[youtube_tool],
    prompt=prompt,
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=[youtube_tool],
    return_intermediate_steps=True,
    verbose=True,  # Keeps terminal tracking on
)

test_question = "Find me a video tutorial about cooking carbonara pasta on YouTube."
    
print("--- Running Isolation Test ---")
response = agent_executor.invoke({
        "input": test_question,
        "chat_history": [],
    })
    
print("\n--- Output Validation ---")
print("Final Output:", response.get("output"))
print("Captured Trajectory Steps:", response.get("intermediate_steps"))

--- Running Isolation Test ---


> Entering new AgentExecutor chain...

Invoking: `youtube_search` with `{'query': {'type': 'string', 'value': 'cooking carbonara pasta'}, 'num_results': 5}`




TypeError: YouTubeSearchTool._run() got an unexpected keyword argument 'num_results'

# since the agent is not able to use the search tool, I will use smolagents since in the HuggingFace, I find that this agent uses smolagents, smolagents[litellm], and youtube-search  packages

In [32]:
from smolagents import CodeAgent, LiteLLMModel, Tool
# Notice the lowercase 's' here
from youtube_search import YoutubeSearch 

# 1. We map the YouTube utility directly to a clean smolagents tool structure
class YouTubeSearchTool(Tool):
    name = "youtube_search"
    description = "Searches YouTube for videos and returns relevant titles and video links."
    inputs = {
        "query": {
            "type": "string",
            "description": "The simple search keywords to pass into YouTube."
        }
    }
    output_type = "string"

    def forward(self, query: str) -> str:
        # .to_dict() returns a direct list of dictionaries
        videos_list = YoutubeSearch(query, max_results=5).to_dict()
        
        videos = []
        for video in videos_list:
            title = video.get("title")
            # The library returns the suffix in 'url_suffix'
            link = f"https://www.youtube.com{video.get('url_suffix')}"
            videos.append(f"- {title}: {link}")
            
        return "\n".join(videos) if videos else "No videos found."

# 2. Wire up the Ollama server connection using LiteLLM layout syntax
model = LiteLLMModel(
    model_id="ollama_chat/qwen2.5:3b-instruct",
    api_base="http://127.0.0.1:11434"
)

youtube_tool = YouTubeSearchTool()

# 3. Spin up the CodeAgent executor
agent = CodeAgent(
    model=model,
    tools=[youtube_tool],
    max_steps=5,
    verbosity_level=0  # This prints every step the local model executes to your console, if we set 2 then we can see what the agent does
)

# 4. Trigger the run instruction
test_question = "Find me a video tutorial about cooking carbonara pasta on YouTube."

task_prompt = f"""You are a video locator assistant. 
Always use the youtube_search tool to find relevant video links before generating your final text.

User Request: {test_question}"""

print("--- Running smolagents Isolation Test ---")
results_websearch = agent.run(task_prompt,
                         return_full_result=True # to get the trajectories and also the cost
                )

--- Running smolagents Isolation Test ---

In [33]:
# get the usage

usage = {
    "input_tokens": 0,
    "output_tokens": 0,
    "total_tokens": 0,
}

for step in results_websearch .steps:
    if isinstance(step, dict):
        step_usage = step.get("token_usage")
        if step_usage:
            for key in usage:
                usage[key] += step_usage.get(key, 0)

print(usage)

{'input_tokens': 4734, 'output_tokens': 483, 'total_tokens': 5217}

In [34]:
# get the tool_call, trajectories
import re
tool_calls = []

for step in results_websearch .steps:
    if not isinstance(step, dict):
        continue

    code = step.get("code_action")
    if not code:
        continue

    matches = re.findall(r'(\w+)\((.*?)\)', code, re.DOTALL)

    for name, args in matches:
        if name != "print":
            tool_calls.append({
                "tool": name,
                "arguments": args
            })

print(tool_calls)

[
    {'tool': 'youtube_search', 'arguments': 'query="Cooking Carbonara Pasta Tutorial"'},
    {'tool': 'final_answer', 'arguments': 'recommendation'}
]

In [ ]:
# ge the final answer
results_websearch.output

'Here are some video tutorials about making carbonara pasta available on YouTube:\n\n- How to Make Classic Carbonara | Jamie Oliver: https://www.youtube.com/watch?v=D_2DBLAt57c&pp=ygUgQ29va2luZyBDYXJib25hcmEgUGFzdGEgVHV0b3JpYWw%3D\n- Can You Really Make Perfect Carbonara in 15 Minutes?: https://www.youtube.com/watch?v=lXEtC3Y9sqY&pp=ygUgQ29va2luZyBDYXJib25hcmEgUGFzdGEgVHV0b3JpYWw%3D\n- Real Spaghetti Carbonara | Antonio Carluccio: https://www.youtube.com/watch?v=3AAdKl1UYZs&pp=ygUgQ29va2luZyBDYXJib25hcmEgUGFzdGEgVHV0b3JpYWw%3D\n- How to make carbonara in 10 minutes!  Perfect pasta every time!: https://www.youtube.com/watch?v=AjX5_XzK0YM&pp=ygUgQ29va2luZyBDYXJib25hcmEgUGFzdGEgVHV0b3JpYWw%3D\n- Gordon Ramsay Cooks Carbonara in Under 10 Minutes | Ramsay in 10: https://www.youtube.com/watch?v=5t7JLjr1FxQ&pp=ygUgQ29va2luZyBDYXJib25hcmEgUGFzdGEgVHV0b3JpYWw%3D\n'

# Use FAQ agent with search tool without a memeory

In [53]:
import sys
import os
import json
import pandas as pd
from rich import print
from minsearch import Index

parent_directory = os.path.dirname(os.getcwd())
sys.path.append(parent_directory)

from ingest import (load_faq_data,
                    build_index,
                    text_search
)
# load the documents, fillter llm_zoocamp ONLY since the ground_truth data is generated from the llm_zoomcamp ONLY for simplicity
documents = load_faq_data()
documents_llm = [doc for doc in documents if doc['course'] == "llm-zoomcamp"]
print(f'The number of documents = {len(documents_llm)}')

# create search index and fit with document (note when I fit, I do not need to use the optimal parameters, only for search I need to use the optimal parameters)
search_index = build_index(documents_llm)

The number of documents = 113

In [66]:
from smolagents import (
    CodeAgent,
    LiteLLMModel, Tool)
# from smolagents import ToolCallingAgent
# 1. Map your tuned text search directly to a clean smolagents tool structure
class FAQSearchTool(Tool):
    name = "text_search_tuned"
    description = (
        "Searches the LLM Zoomcamp FAQ database using optimized keyword weights. "
        "Use this for any course registration, schedules, or prerequisite questions."
    )
    inputs = {
        "query": {
            "type": "string",
            "description": "The simple search keywords or queries to pass into the FAQ database."
        }
    }
    output_type = "string"

    def __init__(self, search_index, **kwargs):
        super().__init__(**kwargs)
        # Pass your existing minsearch index here during instantiation
        self.search_index = search_index 

    def forward(self, query: str) -> str:
        question_boost = 3.0
        section_boost = 0.5
        answer_boost = 10.0
        
        boost_dict = {
            "question": question_boost,
            "section": section_boost,
            "answer": answer_boost,
        }
        filter_dict = {"course": "llm-zoomcamp"}

        # Query your custom search_index
        search_results = self.search_index.search(
            query,
            num_results=5,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )
        
        lines = []
        for doc in search_results:
            lines.append(f"Section: {doc['section']}")
            lines.append(f"Q: {doc['question']}")
            lines.append(f"A: {doc['answer']}\n")
            
        return "\n".join(lines).strip() if lines else "No matching FAQ entries found."

# 2. Wire up the Ollama server connection using LiteLLM layout syntax
model = LiteLLMModel(
    model_id="ollama_chat/qwen2.5:3b-instruct",
    api_base="http://127.0.0.1:11434"
)

# Initialize your tool (Make sure your 'search_index' object is loaded/defined above this in your script)
faq_tool = FAQSearchTool(search_index=search_index)

# 3. Spin up the CodeAgent executor
agent = CodeAgent(
    model=model,
    tools=[faq_tool],
    max_steps=5,
    verbosity_level=0  # This prints every step the local model executes to your console, verbosity_level=2
)

# 4. Trigger the run instruction using your original strict FAQ instructions
# test_question = "Can I get a certificate if I do the projects and homeworks except one since I will go on holiday"
test_question ="Can I join the course?"
# task_prompt = f"""You are a FAQ assistant for the LLM Zoomcamp.

# Always use the text_search_tuned tool before answering.
# Answer ONLY using information returned by the search tool.
# Keep answers concise and directly address the user's question.
# Do not add unrelated information.
# If the answer is not found in the search results, say: "I don't know."

# User Request: {test_question}"""
task_prompt = f"""
You are an FAQ assistant for LLM Zoomcamp.

You have one tool:
text_search_tuned(query)

Follow these steps:

1. Call text_search_tuned with the user's question.
2. Read the returned FAQ information.
3. Give the final answer.

IMPORTANT:
Your final output MUST be inside a code block exactly like this:

<code>
final_answer("your answer here")
</code>

Never write final_answer outside the code block.

User question:
Can I join the course?
"""
print("--- Running smolagents Isolation Test with FAQ Search ---")
search_index_agent_result = agent.run(task_prompt, return_full_result=True )


--- Running smolagents Isolation Test with FAQ Search ---

In [67]:
# get the usage

usage_search_index_agent_result  = {
    "input_tokens": 0,
    "output_tokens": 0,
    "total_tokens": 0,
}

for step in search_index_agent_result.steps:
    if isinstance(step, dict):
        step_usage = step.get("token_usage")
        if step_usage:
            for key in usage:
                usage_search_index_agent_result[key] += step_usage.get(key, 0)

print(usage_search_index_agent_result)

# ge the final answer
print(f'final answer: {search_index_agent_result.output}')

# get the tool_call, trajectories
import re
tool_calls_search_index_agent_result  = []

for step in search_index_agent_result.steps:
    if not isinstance(step, dict):
        continue

    code = step.get("code_action")
    if not code:
        continue

    matches = re.findall(r'(\w+)\((.*?)\)', code, re.DOTALL)

    for name, args in matches:
        if name != "print":
            tool_calls_search_index_agent_result .append({
                "tool": name,
                "arguments": args
            })

print(tool_calls_search_index_agent_result )

{'input_tokens': 5428, 'output_tokens': 60, 'total_tokens': 5488}

final answer: Section: General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after 
submitting your project.

You can only peer-review projects at the time the course is running; after the form is closed and the peer-review 
list is compiled.

Section: Module 1: RAG
Q: OpenAI: Do I have to subscribe and pay for Open AI API for this course?
A: No, you don't have to pay for this service in order to complete the course homeworks. You can use free or 
low-cost alternatives listed in the course GitHub repo.

See the course list of [OpenAI API 
alternatives](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/awesome-llms.md#openai-api-alternatives).

Section: General Course-Related Questions
Q: How should I start the course and follow the weekly workflow?
A: Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the 
(https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub 
repository](https://github.com/DataTalksClub/llm-zoomcamp).

You can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the
(https://courses.datatalks.club/llm-zoomcamp-2026/).

A typical workflow is:

1. Watch the lesson videos.
2. Work through the lesson notebooks/code.
3. Read the homework instructions on GitHub.
4. Submit answers through the course platform before the deadline.

Homework is similar to the lesson flow, but uses a different dataset or slightly different task.

Section: Module 1: RAG
Q: Can I run the course locally instead of Codespaces?
A: Yes. Codespaces is just the easiest way for everyone to start with the same environment.

You can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools
needed for the module.

If you run locally, make sure you document your setup and keep your environment reproducible.

Section: Module 1: RAG
Q: Do I have to use OpenAI, or can I use a different provider?
A: If the provider used in the course isn't available or is blocked in your region (or you simply prefer another 
one), you can use any other LLM provider — the course isn't tied to OpenAI. Just switch to something else:

- Hosted, OpenAI-compatible providers — e.g. Groq, OpenRouter, DeepSeek, Gemini, Z.ai, Mistral. The course code 
uses the OpenAI client, so you usually only need to change the `base_url`, the API key, and the model name.
- Open models via Hugging Face (e.g. Qwen, Llama) if you prefer hosted open-source models.
- Serve a model locally with [Ollama](https://ollama.com/), (https://github.com/vllm-project/vllm), LM Studio, or 
anything else — no external API call at all, so regional blocks don't apply and you don't need a paid key. Most of 
these also expose an OpenAI-compatible endpoint, so the course code works with only a `base_url` change.
- Rent a GPU machine and serve the model there (e.g. with vLLM) if your own machine can't run the model you want. 
This gives you a private OpenAI-compatible endpoint to point the course code at — just remember to stop/delete the 
instance when you're done so you don't keep paying for it.
- A VPN also works if you just need to reach a provider that's blocked at the network level.

Anything with an OpenAI-compatible endpoint (or a locally served model) will work — pick whatever is available and 
convenient for you.

For a curated list of options, see [Awesome 
LLMs](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/awesome-llms.md) in the course repo, which collects 
OpenAI API alternatives and tools for running models locally.

[
    {'tool': 'text_search_tuned', 'arguments': 'query="Can I join the course?"'},
    {'tool': 'final_answer', 'arguments': 'answer'}
]

# Use FAQ agent with search tool with a memeory

In [ ]:
from smolagents import CodeAgent, LiteLLMModel, Tool
from langchain_core.messages import HumanMessage, AIMessage


class FAQSearchTool(Tool):
    name = "text_search_tuned"
    description = (
        "Searches the LLM Zoomcamp FAQ database using optimized keyword weights. "
        "Use this for any course registration, schedules, or prerequisite questions."
    )

    inputs = {
        "query": {
            "type": "string",
            "description": "The simple search keywords or queries to pass into the FAQ database."
        }
    }

    output_type = "string"

    def __init__(self, search_index, **kwargs):
        super().__init__(**kwargs)
        self.search_index = search_index

    def forward(self, query: str) -> str:

        boost_dict = {
            "question": 3.0,
            "section": 0.5,
            "answer": 10.0,
        }

        filter_dict = {
            "course": "llm-zoomcamp"
        }

        search_results = self.search_index.search(
            query,
            num_results=5,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )

        lines = []

        for doc in search_results:
            lines.append(f"Section: {doc['section']}")
            lines.append(f"Q: {doc['question']}")
            lines.append(f"A: {doc['answer']}\n")

        return "\n".join(lines).strip() if lines else "No matching FAQ entries found."


class FAQAgent:

    def __init__(self, search_index):
        # Memory
        self.chat_history = []


        # Model
        self.model = LiteLLMModel(
            model_id="ollama_chat/qwen2.5:3b-instruct",
            api_base="http://127.0.0.1:11434"
        )


        # Tool
        self.faq_tool = FAQSearchTool(
            search_index=search_index
        )


        # Agent
        self.agent = CodeAgent(
            model=self.model,
            tools=[self.faq_tool],
            max_steps=5,
            verbosity_level=0
        )


    def get_history(self):

        if not self.chat_history:
            return "No previous conversation."

        history = []

        for message in self.chat_history:

            if isinstance(message, HumanMessage):
                history.append(
                    f"User: {message.content}"
                )

            elif isinstance(message, AIMessage):
                history.append(
                    f"Assistant: {message.content}"
                )

        return "\n".join(history)



    def update_memory(self, question, answer):

        self.chat_history.append(
            HumanMessage(content=question)
        )

        self.chat_history.append(
            AIMessage(content=answer)
        )

        # keep last 10 messages
        self.chat_history = self.chat_history[-10:]



    def run(self, question):

        history = self.get_history()


        task_prompt = f"""
You are an FAQ assistant for LLM Zoomcamp.

Conversation history:
{history}

You have one tool:
text_search_tuned(query)

Follow these steps:

1. Call text_search_tuned with the user's question.
2. Read the returned FAQ information.
3. Answer ONLY using the returned information.
4. Do not copy the search results.
5. Keep the answer concise.

IMPORTANT:
Your final output MUST be inside a code block exactly like this:

<code>
final_answer("your answer here")
</code>

Never write final_answer outside the code block.

User question:
{question}
"""


        result = self.agent.run(
            task_prompt,
            return_full_result=True
        )


        answer = result.output


        # Update memory
        self.update_memory(
            question,
            answer
        )


        return result

In [70]:
faq_agent_with_memory = FAQAgent(search_index)


question = "Can I join the course?"

search_index_agent_result = faq_agent_with_memory.run(question)


print("Final answer:")
print(search_index_agent_result.output)

Final answer:

Yes, you can join the course. The information provided indicates that the course offers a self-paced option to 
follow along with the course materials without needing to pay extra for the OpenAI API. Certificates are awarded 
upon completion of a live cohort rather than the self-paced mode.

In [71]:
usage_search_index_agent_result = {
    "input_tokens": 0,
    "output_tokens": 0,
    "total_tokens": 0,
}


for step in search_index_agent_result.steps:

    if isinstance(step, dict):

        step_usage = step.get("token_usage")

        if step_usage:
            for key in usage_search_index_agent_result:
                usage_search_index_agent_result[key] += step_usage.get(key, 0)


print(usage_search_index_agent_result)

{'input_tokens': 5444, 'output_tokens': 82, 'total_tokens': 5526}

In [72]:
import re


tool_calls = []


for step in search_index_agent_result.steps:

    if not isinstance(step, dict):
        continue


    code = step.get("code_action")

    if not code:
        continue


    matches = re.findall(
        r'(\w+)\((.*?)\)',
        code,
        re.DOTALL
    )


    for name, args in matches:

        if name != "print":

            tool_calls.append(
                {
                    "tool": name,
                    "arguments": args
                }
            )


print(tool_calls)

[
    {'tool': 'text_search_tuned', 'arguments': 'query="Can I join the course?"'},
    {
        'tool': 'final_answer',
        'arguments': '"Yes, you can join the course. The information provided indicates that the course offers a 
self-paced option to follow along with the course materials without needing to pay extra for the OpenAI API. 
Certificates are awarded upon completion of a live cohort rather than the self-paced mode."'
    }
]